# quiz 1 基礎

In [4]:
from pathlib import Path
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

In [3]:
DATA_ROOT = Path("./dataset/Intel Image Classification")
CLASS_NAMES = [
    "buildings", "forest", "glacier",
    "mountain", "sea", "street"
]
VALID_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

records = []

for split in ["seg_train", "seg_test", "seg_pred"]:
    split_dir = DATA_ROOT / split

    if not split_dir.is_dir():
        raise FileNotFoundError(f"找不到資料夾：{split_dir.resolve()}")

    if split != "seg_pred":
        for name in CLASS_NAMES:
            if not (split_dir / name).is_dir():
                raise FileNotFoundError(f"找不到類別資料夾：{split_dir / name}")

    paths = sorted(
        p for p in split_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in VALID_EXTS
    )

    for path in tqdm(paths, desc=f"檢查 {split}"):
        if split != "seg_pred":
            label = path.relative_to(split_dir).parts[0]
        else:
            label = None

        row = {
            "split": split,
            "path": str(path),
            "label": label,
            "width": None,
            "height": None,
            "mode": None,
            "error": None,
        }

        try:
            with Image.open(path) as img:
                img.load()
                row.update(
                    width=img.width,
                    height=img.height,
                    mode=img.mode,
                )
        except Exception as exc:
            row["error"] = str(exc)

        records.append(row)

image_df = pd.DataFrame(records)

if image_df.empty:
    raise ValueError("沒有找到圖片，請確認DATA_ROOT和副檔名。")

print("圖片總數：", len(image_df))
print("檢查表row/col：", image_df.shape)

print("\n各資料集圖片數：")
display(image_df.groupby("split").size().rename("count").to_frame())

print("\n各類圖片數：")
display(pd.crosstab(image_df["label"], image_df["split"]))

valid_df = image_df[image_df["error"].isna()].copy()

print("\n最常見的圖片尺寸(寬、高)：")
display(valid_df.groupby(["width", "height"]).size().sort_values(ascending=False).head(10).rename("count").to_frame())

print("\n圖片色彩模式：")
display(valid_df["mode"].value_counts().rename("count").to_frame())

bad_df = image_df[image_df["error"].notna()]
print("\n無法讀取的圖片數：", len(bad_df))
display(bad_df[["path", "error"]].head(10))

unexpected = (
    set(image_df["label"].dropna()) - set(CLASS_NAMES)
)
print("\n非預期的類別：", unexpected)

檢查 seg_train:   0%|          | 0/14034 [00:00<?, ?it/s]

檢查 seg_test:   0%|          | 0/3000 [00:00<?, ?it/s]

檢查 seg_pred:   0%|          | 0/7301 [00:00<?, ?it/s]

圖片總數： 24335
檢查表row/col： (24335, 7)

各資料集圖片數：


,count
split,
seg_pred,7301
seg_test,3000
seg_train,14034


...


圖片色彩模式：


,count
mode,
RGB,24335



無法讀取的圖片數： 0


,path,error



非預期的類別： set()


In [5]:
train_pool = valid_df[valid_df["split"] == "seg_train"].copy()
test_df = valid_df[valid_df["split"] == "seg_test"].copy()

train_df, val_df = train_test_split(train_pool, test_size=0.2, random_state=42, stratify=train_pool["label"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 11227
Val: 2807
Test: 3000
